<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/04-spark-sql/01-sql-vs-dataframe-api.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 4.1 — SQL vs DataFrame API in a real script

This notebook builds the same small report three ways — pure SQL, pure
DataFrame, and the recommended mixed shape — and proves the engine
doesn't care.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("4.1-sql-vs-dataframe")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)
orders.createOrReplaceTempView("orders")

## Same report, both languages — and the plan comparison

In [ ]:
sql_version = spark.sql("""
    SELECT category,
           ROUND(SUM(quantity * unit_price), 2) AS revenue,
           COUNT(DISTINCT customer_id)          AS buyers
    FROM orders
    WHERE country IS NOT NULL
    GROUP BY category
""")

df_version = (
    orders
    .filter(col("country").isNotNull())
    .groupBy("category")
    .agg(
        F.round(F.sum(col("quantity") * col("unit_price")), 2).alias("revenue"),
        F.countDistinct("customer_id").alias("buyers"),
    )
)

sql_version.orderBy("category").show()
df_version.orderBy("category").show()

In [ ]:
# The proof there's no performance argument: compare the physical plans
sql_version.explain()
df_version.explain()

## Where DataFrames win: programmatic structure

"Null-count every column" — try writing this as static SQL for an
arbitrary table.

In [ ]:
null_report = orders.select([
    F.count(F.when(col(c).isNull(), 1)).alias(c) for c in orders.columns
])
null_report.show()

## Where SQL wins: dense relational logic over multiple tables

In [ ]:
# A tiny second table, to make the join worth reading
customers = spark.createDataFrame(
    [("c001", "APAC"), ("c002", "EMEA"), ("c003", "AMER"), ("c004", "EMEA"),
     ("c005", "APAC"), ("c006", "AMER"), ("c007", "APAC"), ("c008", "APAC"),
     ("c009", "EMEA"), ("c010", "APAC"), ("c011", "EMEA"), ("c012", "AMER")],
    "customer_id STRING, region STRING",
)
customers.createOrReplaceTempView("customers")

spark.sql("""
    SELECT c.region,
           o.category,
           ROUND(SUM(o.quantity * o.unit_price), 2) AS revenue
    FROM orders o
    JOIN customers c USING (customer_id)
    GROUP BY c.region, o.category
    ORDER BY c.region, revenue DESC
""").show()

## The recommended shape: DF edges, SQL heart, views as interfaces

In [ ]:
QUALITY_RULES = {                       # imagine this arriving from config
    "missing_qty":   "quantity IS NULL",
    "unknown_geo":   "country IS NULL",
    "high_value":    "quantity * unit_price > 100",
}

def load_orders(spark, path):
    df = spark.read.option("header", True).schema(ORDERS_SCHEMA).csv(path)
    return df.dropDuplicates()

def add_quality_flags(df, rules):
    for name, condition in rules.items():
        df = df.withColumn(name, F.expr(condition))
    return df

(
    load_orders(spark, f"{DATA_DIR}/orders.csv")
    .transform(lambda d: add_quality_flags(d, QUALITY_RULES))
    .createOrReplaceTempView("orders_clean")
)

spark.sql("""
    SELECT category,
           COUNT(*)                                   AS orders,
           SUM(CASE WHEN missing_qty THEN 1 ELSE 0 END) AS missing_qty,
           SUM(CASE WHEN high_value  THEN 1 ELSE 0 END) AS high_value
    FROM orders_clean
    GROUP BY category
""").show()

## Exercises

1. Extend `QUALITY_RULES` with a `bulk_order` rule (quantity >= 3) WITHOUT touching the functions — the point of config-driven structure. Re-run.
2. Take the region/category SQL report and split its natural seam: materialize the joined+enriched rows as a view built by a `def enrich_orders(orders_df, customers_df)` function, keep only the GROUP BY in SQL.
3. Write `add_quality_flags` badly on purpose — as an f-string SQL generator (`f"SELECT *, {cond} AS {name} FROM ..."`). Get it working, then write one sentence on what made it worse (quoting? nesting? testing?).
4. Unit-test `add_quality_flags` with `spark.createDataFrame` fixture rows: one row per rule, assert the flags. (Preview of Module 16's whole methodology.)

In [ ]:
# your exercise code here
